# 📊 CollectiveLens - Complete Document Intelligence System

**NGO Knowledge Base with AI Summaries & Advanced Analytics**

## ✨ Complete Features:
- 93 multilingual policy documents
- 8 policy categories with zero-shot classification
- **English AI summaries** (Claude API) - all languages → English
- **Relevance explanations** - why documents match searches
- **Complete analytics dashboard** - Overview + Heatmap + Distribution
- Multilingual semantic search (SentenceTransformers)
- KeyBERT semantic clustering
- Interactive Gradio interface

---

## 📋 Cell Structure:

1. **Dependencies** - Install packages (2 min)
2. **PDF + Summaries** - Extract text & generate English summaries (~20 min first time)
3. **Embeddings** - Multilingual semantic embeddings (3 min)
4. **Classification** - Categories + KeyBERT clustering (5 min)
5. **Launch Gradio** - Full interface with all features
6. **Quick Relaunch** - Fast relaunch (30 seconds)

---

## 🔄 To Regenerate Summaries:


In [ ]:
!rm raw_documents.json
!rm documents_with_clusters.csv

In [ ]:
##APIKEY

In [8]:
import os, getpass

if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key (not saved in the notebook): ")

In [ ]:
import boto3

s3 = boto3.client("s3")
print([b["Name"] for b in s3.list_buckets()["Buckets"]])

In [ ]:
#DocfromSG

In [ ]:
import os
import boto3

BUCKET = os.getenv("COLLECTIVELENS_S3_BUCKET", "YOUR-S3-BUCKET")   # optional: only if your PDFs are in S3
PREFIX = "pdf/"
LOCAL_DIR = "Doc Project"

os.makedirs(LOCAL_DIR, exist_ok=True)

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

downloaded = 0

for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if not key.lower().endswith(".pdf"):
            continue

        fname = os.path.basename(key)
        local_path = os.path.join(LOCAL_DIR, fname)

        if not os.path.exists(local_path):
            s3.download_file(BUCKET, key, local_path)
            downloaded += 1

print(f"Downloaded {downloaded} PDFs from S3 into '{LOCAL_DIR}/'")


---
# 📦 CELL 1: Dependencies

Install all required packages.

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Suppress warning

!pip install sentence-transformers --break-system-packages --quiet
!pip install keybert --break-system-packages --quiet
!pip install anthropic --upgrade --break-system-packages --quiet
!pip install pdfplumber --break-system-packages --quiet
!pip install langdetect --break-system-packages --quiet
!pip install gradio --break-system-packages --quiet
%pip install -U gradio


print("✓ All dependencies installed")

In [ ]:
!pip install -U sentence-transformers scikit-learn anthropic matplotlib pandas numpy --no-cache-dir --break-system-packages

In [ ]:
!pip install gradio --break-system-packages

---
# 📄 CELL 2: PDF Extraction + English AI Summaries

**Extracts text and generates English summaries using Claude API.**

⏰ **Time:** ~15-20 minutes (93 documents)

💰 **Cost:** ~$2-3 (Claude Haiku)

**Skips if `raw_documents.json` exists.**

**To force regeneration:** `!rm raw_documents.json`

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
CLAUDE_MODEL = os.getenv("COLLECTIVELENS_CLAUDE_MODEL", "claude-haiku-4-5-20251001")  # claude-3-haiku and claude-3-5-haiku are retired

import pdfplumber
import pandas as pd
import numpy as np
from pathlib import Path
from langdetect import detect
import json
import warnings
import signal
import gc
from contextlib import contextmanager
from anthropic import Anthropic
import time
import re

warnings.filterwarnings('ignore')

print("="*80)
print("COLLECTIVELENS - PDF EXTRACTION WITH AI SUMMARIES")
print("="*80)
print()

os.chdir(os.getenv("COLLECTIVELENS_DIR", os.getcwd()))

DOCS_PATH = Path('Doc Project/')
METADATA_PATH = Path('metadatatable.csv')
RAW_DOCS_FILE = Path('raw_documents.json')

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise RuntimeError("ANTHROPIC_API_KEY not set in environment")

print(f"Working directory: {os.getcwd()}")
print()

# ------------------------------------------------------------------
# Claude summary
# ------------------------------------------------------------------

def get_claude_summary(text, title=""):
    client = Anthropic(api_key=ANTHROPIC_API_KEY)
    prompt = f"""Analyze this policy document.

Title: {title}
Text: {text[:8000]}

Provide a 2-3 sentence summary in English.
Plain text only."""
    try:
        response = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=200,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text.strip()
    except Exception as e:
        print(f"    ⚠️ Summary failed: {str(e)[:100]}")
        return "Summary unavailable"

def safe_detect(text):
    if not text or len(text.strip()) < 10:
        return "unknown"
    try:
        return detect(text)
    except:
        return "unknown"

# ------------------------------------------------------------------
# Timeout handling
# ------------------------------------------------------------------

class TimeoutError(Exception):
    pass

@contextmanager
def timeout(seconds):
    def signal_handler(signum, frame):
        raise TimeoutError("Timeout")
    old = signal.signal(signal.SIGALRM, signal_handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old)

# ------------------------------------------------------------------
# Load checkpoint if exists
# ------------------------------------------------------------------

existing_docs = []
processed_filenames = set()

if RAW_DOCS_FILE.exists():
    print("✓ Found existing raw_documents.json")
    try:
        with open(RAW_DOCS_FILE, 'r', encoding='utf-8') as f:
            existing_docs = json.load(f)
        processed_filenames = {doc['filename'] for doc in existing_docs}
        print(f"✓ Already processed: {len(existing_docs)} documents")
        print("📋 Will resume from where it stopped")
    except:
        print("⚠️ Could not load existing file, starting fresh")
        existing_docs = []
    print()

# ------------------------------------------------------------------
# PDF extraction
# ------------------------------------------------------------------

def extract_text_from_pdf(pdf_path):
    text_parts = []
    try:
        with timeout(30):
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    try:
                        t = page.extract_text()
                        if t:
                            text_parts.append(t)
                    except:
                        continue
    except:
        pass
    return " ".join(text_parts)

# ------------------------------------------------------------------
# Metadata loading (canonical)
# ------------------------------------------------------------------

print("Loading metadata...")

def normalize_key(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().replace("\\", "/")
    x = x.split("/")[-1]
    x = re.sub(r"\.pdf$", "", x, flags=re.IGNORECASE)
    return x

# Load metadata
metadata_df = pd.read_csv(METADATA_PATH, engine="python", sep=None)

# Drop junk columns
metadata_df = metadata_df.loc[
    :, ~metadata_df.columns.astype(str).str.startswith("Unnamed")
].copy()

# Create join key FIRST
metadata_df["_join_key"] = metadata_df["file_name"].map(normalize_key)

# Optional diagnostic (safe now)
n_dups = (metadata_df["_join_key"].value_counts() > 1).sum()
print(f"⚠️ Metadata duplicates (same file_name): {n_dups}")

# Build lookup (deduplicate deterministically)
meta_by_key = (
    metadata_df
    .drop_duplicates("_join_key", keep="first")
    .set_index("_join_key")
    .to_dict(orient="index")
)

print(f"✓ Loaded metadata: {metadata_df.shape}")
print(f"Columns: {metadata_df.columns.tolist()}")
print()

# ------------------------------------------------------------------
# Process PDFs
# ------------------------------------------------------------------

pdf_files = list(DOCS_PATH.glob("*.pdf"))
print(f"✓ Found {len(pdf_files)} total PDF files")
print(f"⏭️  Skipping {len(processed_filenames)} already processed")
print(f"🔄 Processing {len(pdf_files) - len(processed_filenames)} remaining")
print()

import json

def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

JSONL_PATH = Path("raw_documents.jsonl")

processed = set()
if JSONL_PATH.exists():
    with open(JSONL_PATH, "r", encoding="utf-8") as f:
        for line in f:
            try:
                processed.add(json.loads(line)["filename"])
            except:
                continue

print(f"Already processed: {len(processed)}")

for i, pdf_path in enumerate(pdf_files, 1):
    if pdf_path.name in processed:
        print(f"[{i}/{len(pdf_files)}] {pdf_path.name[:40]}... ✓ Already done")
        continue

    print(f"[{i}/{len(pdf_files)}] {pdf_path.name[:40]}...")

    text = extract_text_from_pdf(pdf_path)
    if not text or len(text.strip()) < 100:
        print("  ⊗ Skip (no text)")
        continue

    join_key = normalize_key(pdf_path.name)
    m = meta_by_key.get(join_key, {})

    title = m.get("title", pdf_path.stem)
    org   = m.get("member_org", "Unknown")
    year  = m.get("publication_year", None)

    try:
        year = int(year) if year not in [None, "", 0, "0"] else None
    except:
        year = None

    summary = get_claude_summary(text, title)
    language = safe_detect(text)

    doc = {
        "filename": pdf_path.name,
        "title": title,
        "text": text,
        "summary": summary,
        "language": language,
        "member_org": org,
        "publication_year": year,
        "char_count": len(text)
    }

    append_jsonl(JSONL_PATH, doc)
    print("  ✓ Saved")

    if i % 5 == 0:
        gc.collect()
        time.sleep(1)

print()
print("COMPLETE: extraction finished (streamed to raw_documents.jsonl)")
print("Next step: convert raw_documents.jsonl → raw_documents.json")
print("="*80)
print("CELL COMPLETE")
print("="*80)

In [10]:
import json

records = []
with open("raw_documents.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

with open("raw_documents.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print("Converted JSONL → raw_documents.json:", len(records))

Converted JSONL → raw_documents.json: 93


---
# 🧮 CELL 3: Embeddings

In [11]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from sentence_transformers import SentenceTransformer
import json
import pandas as pd
import numpy as np
import gc
from pathlib import Path

print("="*80)
print("EMBEDDINGS")
print("="*80 + "\n")

MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
EMB_FILE = Path('doc_embeddings.npy')

if EMB_FILE.exists():
    embeddings = np.load(EMB_FILE)
    print(f"✓ Loaded: {embeddings.shape}")
else:
    with open('raw_documents.json', 'r', encoding='utf-8') as f:
        docs = json.load(f)
    df = pd.DataFrame(docs)
    print(f"✓ {len(df)} docs\n")
    
    model = SentenceTransformer(MODEL_NAME)
    print("✓ Model loaded\n")
    
    all_emb = []
    for i in range(0, len(df), 50):
        chunk = df['text'].iloc[i:min(i+50, len(df))].tolist()
        emb = model.encode(chunk, batch_size=4, show_progress_bar=False)
        all_emb.append(emb)
        print(f"Chunk {i//50+1}: ✓")
    
    doc_embeddings = np.vstack(all_emb)
    np.save(EMB_FILE, doc_embeddings)
    print(f"\n✓ Saved: {doc_embeddings.shape}")
    del model, all_emb
    gc.collect()

print("\n" + "="*80)
print("CELL 3 COMPLETE")
print("="*80)

2025-12-16 15:19:28.464248: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


EMBEDDINGS

✓ Loaded: (93, 384)

CELL 3 COMPLETE


In [12]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import numpy as np
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from keybert import KeyBERT
import gc

print("="*80)
print("CLASSIFICATION + KEYBERT")
print("="*80 + "\n")

MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
CLUSTERS_FILE = Path('documents_with_clusters.csv')

CATEGORIES = {
    'Higher Education (incl. Civic Education)': 'higher education, university, academic, civic education, citizenship',
    'Human Rights': 'human rights, equality, discrimination, inclusion, diversity, LGBTIQ+',
    'Climate Justice': 'climate change, environmental, sustainability, green transition',
    'Youth Rights & Participation': 'youth participation, youth policy, youth rights, young people',
    'Peace & Security': 'peace education, conflict resolution, security, peacebuilding',
    'Mobility / Erasmus': 'student mobility, Erasmus, international exchange, study abroad',
    'EU Funding & Governance': 'EU funding, European policy, EU governance, EU institutions',
    'Mental Health & Wellbeing': 'mental health, wellbeing, psychological support'
}

if CLUSTERS_FILE.exists():
    print("✓ Found complete data")
    df = pd.read_csv(CLUSTERS_FILE)
    print(f"✓ {len(df)} docs loaded")
else:
    with open('raw_documents.json', 'r') as f:
        docs = json.load(f)
    df = pd.DataFrame(docs)
    doc_embeddings = np.load('doc_embeddings.npy')
    print(f"✓ {len(df)} docs | {doc_embeddings.shape}\n")
    
    model = SentenceTransformer(MODEL_NAME)
    
    # Classification
    print("Classification...")
    cat_names = list(CATEGORIES.keys())
    cat_emb = model.encode(list(CATEGORIES.values()), show_progress_bar=False)
    sims = cosine_similarity(doc_embeddings, cat_emb)
    df['topic_category'] = [cat_names[i] for i in sims.argmax(axis=1)]
    df['topic_confidence'] = sims.max(axis=1)
    print("✓\n")
    print(df['topic_category'].value_counts())
    print()
    
    # Clustering
    print("Clustering...")
    kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
    df['cluster'] = kmeans.fit_predict(doc_embeddings)
    print("✓\n")
    
    # KeyBERT
    print("KeyBERT keywords...")
    kw_model = KeyBERT(model=model)
    cluster_kws = []
    for cid in sorted(df['cluster'].unique()):
        texts = df[df['cluster'] == cid]['text'].tolist()
        if len(texts) < 2:
            continue
        combined = ' '.join([t[:2000] for t in texts[:10]])
        try:
            kws = kw_model.extract_keywords(combined, keyphrase_ngram_range=(1,2), stop_words='english', top_n=12, use_mmr=True, diversity=0.5)
            terms = [k[0] for k in kws]
            cluster_kws.append({'cluster': cid, 'top_terms': ', '.join(terms)})
            print(f"  C{cid}: {', '.join(terms[:5])}...")
        except:
            pass
    print(f"\n✓ {len(cluster_kws)} clusters\n")
    
    pd.DataFrame(cluster_kws).to_csv('cluster_top_terms.csv', index=False)
    df['publication_year'] = df['publication_year'].fillna(0).astype(int)
    df.to_csv(CLUSTERS_FILE, index=False)
    print(f"✓ Saved {CLUSTERS_FILE}")
    
    del model, cat_emb, kw_model
    gc.collect()

print("\n" + "="*80)
print("CELL 4 COMPLETE")
print("="*80)

CLASSIFICATION + KEYBERT

✓ 93 docs | (93, 384)

Classification...
✓

topic_category
EU Funding & Governance                     32
Youth Rights & Participation                23
Mobility / Erasmus                          12
Peace & Security                            11
Climate Justice                              8
Higher Education (incl. Civic Education)     4
Mental Health & Wellbeing                    2
Human Rights                                 1
Name: count, dtype: int64

Clustering...
✓

KeyBERT keywords...
  C0: work platform, arbeitsmarkt außerschulische, eesti noorsootöötajate, position paper, forum marginalised...
  C1: europe recommendations, lifelong learning, european youth, 2025 researched, languages competences...
  C2: climate action, sustainable development, global crisis, contributes eradicating, said policy...
  C3: solidarity policy, policy paper, freedom institutional, sustainability chapter, 10 responsibility...
  C4: jugendlichen österreichs, 2011 liechtens

---
# 🎯 CELL 4: Classification + KeyBERT

---
# 🚀 CELL 5: Launch Gradio - COMPLETE INTERFACE

**Full interface with:**
- English summaries
- Relevance explanations
- Complete analytics dashboard (Overview + Heatmap + Distribution)
- All filters working

In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
CLAUDE_MODEL = os.getenv("COLLECTIVELENS_CLAUDE_MODEL", "claude-haiku-4-5-20251001")  # claude-3-haiku and claude-3-5-haiku are retired

import gradio as gr
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from io import BytesIO
import base64

# Optional deps (robust)
try:
    from sentence_transformers import SentenceTransformer
except Exception:
    SentenceTransformer = None

try:
    from anthropic import Anthropic
except Exception:
    Anthropic = None


print("="*80)
print("🚀 LAUNCHING COLLECTIVELENS (UX DEMO VERSION)")
print("="*80 + "\n")

# =========================
# Load data + models
# =========================
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

df = pd.read_csv("documents_with_clusters.csv")
doc_embeddings = np.load("doc_embeddings.npy")

if SentenceTransformer is None:
    raise ModuleNotFoundError(
        "sentence_transformers not found. Install it in this environment, then re-run:\n"
        "pip install -U sentence-transformers"
    )
model = SentenceTransformer(MODEL_NAME)

# "Read full document" support (optional columns)
HAS_PDF_URL = "pdf_url" in df.columns
HAS_PDF_PATH = "pdf_path" in df.columns

# Dropdown choices
topic_list = sorted(df["topic_category"].dropna().unique().tolist()) if "topic_category" in df.columns else []
org_list = sorted(df["member_org"].dropna().unique().tolist()) if "member_org" in df.columns else []

# Year range
if "publication_year" in df.columns:
    py = pd.to_numeric(df["publication_year"], errors="coerce").fillna(0)
    years_series = df.loc[py > 0, "publication_year"]
    year_min = int(pd.to_numeric(years_series, errors="coerce").min()) if not years_series.empty else 2000
    year_max = int(pd.to_numeric(years_series, errors="coerce").max()) if not years_series.empty else 2025
else:
    year_min, year_max = 2000, 2025


# =========================
# Helpers
# =========================
def plot_to_html(fig) -> str:
    buf = BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=140)
    buf.seek(0)
    img_data = base64.b64encode(buf.read()).decode()
    plt.close(fig)
    return f"<img src='data:image/png;base64,{img_data}' style='max-width:100%;height:auto;margin:14px 0 6px 0;'/>"

def clamp_text(s: str, n: int) -> str:
    s = (s or "").strip()
    return s[:n] + ("…" if len(s) > n else "")

def safe_int_year(x):
    try:
        y = int(float(x))
        return y if y > 0 else None
    except Exception:
        return None

def get_effective_api_key(ui_key: str | None) -> str | None:
    # Prefer env var; fallback to user-provided key from UI
    env_key = os.getenv("ANTHROPIC_API_KEY")
    if env_key and env_key.strip():
        return env_key.strip()
    if ui_key and ui_key.strip():
        return ui_key.strip()
    return None

def get_relevance(api_key: str | None, keyword: str, text: str, title: str, score: float) -> str | None:
    if not api_key or not keyword or score < 0.30:
        return None
    if Anthropic is None:
        return None

    client = Anthropic(api_key=api_key)
    try:
        resp = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=120,
            messages=[{
                "role": "user",
                "content": (
                    f'Explain in ONE sentence why the document "{title}" is relevant to "{keyword}". '
                    f'Start exactly with: "This document is relevant because..." '
                    f'Text excerpt: {text[:2500]}'
                )
            }]
        )
        return resp.content[0].text.strip()
    except Exception:
        return None


# =========================
# Search logic
# =========================
def search_documents(keyword="", topic="All Topics", org="All Organizations", year_range=None, sort_by="Highest Relevance"):
    results = df.copy()

    # Similarity
    if keyword and keyword.strip():
        query_emb = model.encode([keyword])
        sims = cosine_similarity(query_emb, doc_embeddings)[0]
        results["similarity"] = sims
        results = results.sort_values("similarity", ascending=False)
    else:
        results["similarity"] = 0.0

    # Topic
    if "topic_category" in results.columns and topic != "All Topics":
        results = results[results["topic_category"] == topic]

    # Org
    if "member_org" in results.columns and org and org != "All Organizations":
        if isinstance(org, list):
            if "All Organizations" not in org:
                results = results[results["member_org"].isin(org)]
        else:
            results = results[results["member_org"] == org]

    # Year range
    if year_range is not None and "publication_year" in results.columns:
        y0, y1 = year_range
        py = pd.to_numeric(results["publication_year"], errors="coerce")
        results = results[py.notna() & (py >= y0) & (py <= y1)]

    # Sort
    if sort_by == "Most Recent" and "publication_year" in results.columns:
        py = pd.to_numeric(results["publication_year"], errors="coerce")
        results = results.assign(_py=py).sort_values("_py", ascending=False).drop(columns=["_py"], errors="ignore")

    return results.head(20)


# =========================
# Period selection
# =========================
def period_to_year_range(period_choice: str, year_from: str, year_to: str):
    if period_choice == "All years":
        return None, False
    if period_choice == "Last 3 years":
        return [max(year_min, year_max - 2), year_max], False
    if period_choice == "Last 5 years":
        return [max(year_min, year_max - 4), year_max], False
    if period_choice == f"{year_max-5}–{year_max}":
        return [max(year_min, year_max - 5), year_max], False
    if period_choice == "Custom range":
        yf = year_min if year_from == "All" else int(year_from)
        yt = year_max if year_to == "All" else int(year_to)
        return [min(yf, yt), max(yf, yt)], True
    return None, False


# =========================
# HTML card formatting (NO DUPLICATED SUMMARY)
# =========================
def format_results(results_df: pd.DataFrame, keyword: str, api_key: str | None) -> str:
    if results_df.empty:
        return (
            "<div class='cl-empty'>"
            "<div class='cl-empty-title'>No documents found</div>"
            "<div class='cl-empty-sub'>Try removing filters or using a broader keyword.</div>"
            "</div>"
        )

    # Precompute relevance for top 5 (predictable latency)
    relevance = {}
    if keyword and keyword.strip() and api_key:
        for idx, row in results_df.head(5).iterrows():
            sim = float(row.get("similarity", 0) or 0)
            if sim >= 0.30:
                rel = get_relevance(
                    api_key=api_key,
                    keyword=keyword,
                    text=row.get("text", "") or "",
                    title=row.get("title", "") or "",
                    score=sim,
                )
                if rel:
                    relevance[idx] = rel

    html = (
        "<div class='cl-topbar'>"
        f"<div class='cl-topbar-title'>Found {len(results_df)} documents</div>"
        "<div class='cl-topbar-sub'>Semantic search + AI summaries</div>"
        "</div>"
    )

    for idx, row in results_df.iterrows():
        title = (row.get("title", "Untitled") or "Untitled").strip()
        topic = (row.get("topic_category", "Unknown") or "Unknown").strip()
        org = (row.get("member_org", "Unknown") or "Unknown").strip()
        year = safe_int_year(row.get("publication_year", None))
        sim = float(row.get("similarity", 0) or 0)
        summary = (row.get("summary", "") or "").strip()
        full_text = (row.get("text", "") or "").strip()

        # link for PDF
        pdf_link = None
        if HAS_PDF_URL:
            v = row.get("pdf_url", None)
            if isinstance(v, str) and v.strip():
                pdf_link = v.strip()
        elif HAS_PDF_PATH:
            v = row.get("pdf_path", None)
            if isinstance(v, str) and v.strip():
                pdf_link = v.strip()

        tags = []
        tags.append(f"<span class='pill pill-topic'>📚 {topic}</span>")
        tags.append(f"<span class='pill pill-org'>🏢 {org}</span>")
        if year:
            tags.append(f"<span class='pill pill-year'>📅 {year}</span>")
        if sim > 0:
            tags.append(f"<span class='pill pill-match'>✓ {int(sim*100)}% match</span>")

        # Summary (single block, expandable, no duplication)
        summary_full = summary if summary else "(No summary available)"
        summary_short = clamp_text(summary_full, 320)

        summary_html = f"""
        <details class="cl-summary">
          <summary class="cl-summary-head">
            <div class="cl-summary-label">📝 Summary</div>
            <div class="cl-summary-short">{summary_short}</div>
            <div class="cl-summary-toggle">
              <span class="cl-more">Read more</span>
              <span class="cl-less">Show less</span>
            </div>
          </summary>
          <div class="cl-summary-full">{summary_full}</div>
        </details>
        """

        # Why relevant (only if we have it; otherwise keep button disabled-looking but clean)
        if idx in relevance:
            why_html = f"""
            <details class="cl-why">
              <summary class="cl-btn cl-btn-secondary">💡 Why is this relevant?</summary>
              <div class="cl-callout cl-callout-why">{relevance[idx]}</div>
            </details>
            """
        else:
            # show as disabled ONLY if keyword exists but key missing; otherwise hide completely
            if keyword and keyword.strip() and not api_key:
                why_html = "<span class='cl-btn cl-btn-disabled' title='Set ANTHROPIC_API_KEY (or paste it above) to enable explanations'>💡 Why is this relevant?</span>"
            else:
                why_html = ""

        # Read full doc CTA
        if pdf_link:
            read_doc_btn = f"<a class='cl-btn cl-btn-primary' target='_blank' rel='noopener' href='{pdf_link}'>📄 Read full document</a>"
        else:
            read_doc_btn = "<span class='cl-btn cl-btn-disabled' title='No document link available (add pdf_url or pdf_path in the CSV)'>📄 Read full document</span>"

        # Optional: quick "Read extracted text" (if you want a fallback viewer without PDFs)
        # You can remove this if you don't want it.
        extracted_text_html = ""
        if full_text:
            extracted_text_short = clamp_text(full_text, 500)
            extracted_text_html = f"""
            <details class="cl-why">
              <summary class="cl-btn cl-btn-tertiary">📃 Read extracted text</summary>
              <div class="cl-callout cl-callout-summary" style="white-space:pre-wrap;">{extracted_text_short}</div>
            </details>
            """

        html += (
            "<div class='cl-card'>"
            f"<div class='cl-title'>{title}</div>"
            f"<div class='cl-tags'>{''.join(tags)}</div>"
            f"{summary_html}"
            "<div class='cl-actions'>"
            f"{why_html}"
            f"{read_doc_btn}"
            f"{extracted_text_html}"
            "</div>"
            "</div>"
        )

    return html


# =========================
# Analytics
# =========================
def generate_stats(org_filter: list | None) -> str:
    filtered = df.copy()

    if "member_org" in filtered.columns and org_filter and "All Organizations" not in org_filter:
        filtered = filtered[filtered["member_org"].isin(org_filter)]

    if filtered.empty:
        return "<div class='cl-empty'><div class='cl-empty-title'>No data</div></div>"

    n_docs = len(filtered)
    n_orgs = filtered["member_org"].nunique() if "member_org" in filtered.columns else 0
    n_topics = filtered["topic_category"].nunique() if "topic_category" in filtered.columns else 0
    n_langs = filtered["language"].nunique() if "language" in filtered.columns else 0
    avg_conf = filtered["topic_confidence"].mean() if "topic_confidence" in filtered.columns else 0
    n_clusters = filtered["cluster"].nunique() if "cluster" in filtered.columns else 0

    html = (
        "<div class='cl-metrics'>"
        f"<div class='metric metric-a'><div class='metric-num'>{n_docs}</div><div class='metric-label'>📄 Total Documents</div></div>"
        f"<div class='metric metric-b'><div class='metric-num'>{n_orgs}</div><div class='metric-label'>🏢 Organizations</div></div>"
        f"<div class='metric metric-c'><div class='metric-num'>{n_topics}</div><div class='metric-label'>📋 Policy Areas</div></div>"
        f"<div class='metric metric-d'><div class='metric-num'>{n_langs}</div><div class='metric-label'>🌍 Languages</div></div>"
        f"<div class='metric metric-e'><div class='metric-num'>{int(avg_conf*100)}%</div><div class='metric-label'>🎯 Avg Confidence</div></div>"
        f"<div class='metric metric-f'><div class='metric-num'>{n_clusters}</div><div class='metric-label'>🔮 AI Clusters</div></div>"
        "</div>"
    )

    if "topic_category" in filtered.columns:
        fig1, ax1 = plt.subplots(figsize=(12, 5))
        counts = filtered["topic_category"].value_counts()
        ax1.barh(counts.index, counts.values)
        ax1.set_title("Documents per Policy Area")
        ax1.set_xlabel("Documents")
        ax1.set_ylabel("Policy Area")
        plt.tight_layout()
        html += "<h3 class='cl-h3'>📊 Documents per Policy Area</h3>"
        html += plot_to_html(fig1)

    if "publication_year" in filtered.columns and "topic_category" in filtered.columns:
        py = pd.to_numeric(filtered["publication_year"], errors="coerce")
        valid = filtered[py.notna() & (py > 0)].copy()
        if not valid.empty:
            valid["publication_year"] = pd.to_numeric(valid["publication_year"], errors="coerce").astype(int)
            heatmap_data = pd.crosstab(valid["topic_category"], valid["publication_year"]).fillna(0)

            years_sorted = sorted(heatmap_data.columns.tolist())
            topics_sorted = heatmap_data.index.tolist()

            fig2, ax2 = plt.subplots(figsize=(13, 6))
            c = ax2.imshow(heatmap_data[years_sorted].values, aspect="auto")
            ax2.set_xticks(range(len(years_sorted)))
            ax2.set_xticklabels(years_sorted, rotation=45, ha="right", fontsize=8)
            ax2.set_yticks(range(len(topics_sorted)))
            ax2.set_yticklabels(topics_sorted, fontsize=8)
            ax2.set_title("Policy Areas Over Time")
            ax2.set_xlabel("Year")
            ax2.set_ylabel("Policy Area")
            fig2.colorbar(c, ax=ax2, label="Documents")
            plt.tight_layout()

            html += "<h3 class='cl-h3'>📅 Policy Areas Over Time</h3>"
            html += plot_to_html(fig2)

    return html


# =========================
# Gradio UI (redesigned)
# =========================
CSS = """
.gradio-container { max-width: 1400px !important; }

.cl-hero {
  background: linear-gradient(135deg,#667eea 0%,#764ba2 100%);
  padding: 28px;
  border-radius: 18px;
  text-align: center;
  margin-bottom: 18px;
  box-shadow: 0 12px 28px rgba(0,0,0,0.10);
}
.cl-hero h1 {
  color: white;
  margin: 0;
  font-size: 44px;
  font-weight: 900;
  letter-spacing: -0.8px;
}
.cl-hero p {
  color: rgba(255,255,255,0.95);
  margin: 8px 0 0;
  font-size: 16px;
  font-weight: 600;
}

.cl-topbar {
  background: #f9fafb;
  border: 1px solid #e5e7eb;
  padding: 14px 16px;
  border-radius: 14px;
  margin: 14px 0 14px 0;
}
.cl-topbar-title { font-size: 16px; font-weight: 900; color: #111827; }
.cl-topbar-sub { font-size: 12.5px; color: #6b7280; margin-top: 2px; }

.cl-card {
  background: white;
  border: 1px solid #e5e7eb;
  border-radius: 16px;
  padding: 16px 16px 14px 16px;
  margin-bottom: 14px;
  box-shadow: 0 1px 2px rgba(0,0,0,0.04);
}
.cl-title {
  font-size: 17px;
  font-weight: 900;
  color: #111827;
  line-height: 1.25;
  margin-bottom: 10px;
}
.cl-tags { display: flex; gap: 8px; flex-wrap: wrap; margin-bottom: 10px; }
.pill {
  display: inline-flex;
  align-items: center;
  padding: 4px 10px;
  border-radius: 999px;
  font-size: 12.5px;
  font-weight: 800;
  border: 1px solid transparent;
}
.pill-topic { background:#ede9fe; color:#5b21b6; border-color:#ddd6fe; }
.pill-org   { background:#dbeafe; color:#1e40af; border-color:#bfdbfe; }
.pill-year  { background:#fef3c7; color:#92400e; border-color:#fde68a; }
.pill-match { background:#dcfce7; color:#166534; border-color:#bbf7d0; }

.cl-actions {
  display: flex;
  gap: 10px;
  flex-wrap: wrap;
  align-items: center;
  margin-top: 12px;
}

/* Button styles */
.cl-btn {
  display: inline-flex;
  align-items: center;
  justify-content: center;
  gap: 6px;
  padding: 8px 12px;
  border-radius: 12px;
  font-size: 13px;
  font-weight: 900;
  cursor: pointer;
  user-select: none;
  border: 1px solid transparent;
  white-space: nowrap;
}
.cl-btn-primary {
  background: #f97316;
  border-color: #fb923c;
  color: white !important;
  text-decoration: none !important;
}
.cl-btn-secondary {
  background: #fff7ed;
  border-color: #fed7aa;
  color: #7c2d12;
}
.cl-btn-tertiary {
  background: #f8fafc;
  border-color: #e2e8f0;
  color: #0f172a;
}
.cl-btn-disabled {
  background: #f3f4f6;
  border-color: #e5e7eb;
  color: #9ca3af;
  cursor: not-allowed;
}

/* Callouts */
.cl-callout {
  margin-top: 8px;
  border-radius: 14px;
  padding: 12px 12px;
  font-size: 13.3px;
  line-height: 1.55;
}
.cl-callout-summary {
  background: #ffffff;
  border: 1px solid #e5e7eb;
  color: #111827;
}
.cl-callout-why {
  background: #fff7ed;
  border: 1px solid #fed7aa;
  color: #7c2d12;
}

/* Metrics */
.cl-metrics {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
  gap: 12px;
  margin: 14px 0 18px 0;
}
.metric {
  border-radius: 16px;
  padding: 16px;
  border: 1px solid rgba(0,0,0,0.06);
  text-align: center;
}
.metric-num { font-size: 34px; font-weight: 950; margin-bottom: 4px; }
.metric-label { font-size: 12.5px; font-weight: 900; color: #334155; }

.metric-a { background:#eef2ff; border-color:#c7d2fe; color:#1f2a5a; }
.metric-b { background:#fce7f3; border-color:#fbcfe8; color:#7a1f4a; }
.metric-c { background:#e0f2fe; border-color:#bae6fd; color:#0b4a6f; }
.metric-d { background:#ffedd5; border-color:#fed7aa; color:#7c2d12; }
.metric-e { background:#ecfdf5; border-color:#bbf7d0; color:#14532d; }
.metric-f { background:#fee2e2; border-color:#fecaca; color:#7f1d1d; }

.cl-h3 { margin-top: 16px; margin-bottom: 8px; font-weight: 950; color:#111827; }

/* Empty */
.cl-empty {
  border: 1px dashed #cbd5e1;
  background: #f8fafc;
  border-radius: 16px;
  padding: 28px;
  text-align: center;
}
.cl-empty-title { font-weight: 950; color:#0f172a; font-size: 16px; }
.cl-empty-sub { margin-top: 6px; color:#64748b; font-size: 13px; }

/* Footer */
.cl-footer {
  text-align: center;
  padding: 16px 10px;
  color: #6b7280;
  border-top: 1px solid #e5e7eb;
  margin-top: 18px;
  font-size: 13px;
}
.cl-footer strong { color:#111827; }

/* Summary: ONE block, expand in place, no duplication */
.cl-summary {
  background: #f0f9ff;
  border: 1px solid #bae6fd;
  border-radius: 16px;
  padding: 12px 12px;
}
.cl-summary summary { list-style:none; cursor:pointer; }
.cl-summary summary::-webkit-details-marker { display:none; }

.cl-summary-head { display:flex; flex-direction:column; gap:6px; }
.cl-summary-label { font-weight:950; color:#075985; font-size:12.5px; }
.cl-summary-short { color:#0c4a6e; font-size:13.5px; line-height:1.6; }
.cl-summary-toggle { font-weight:900; font-size:13px; color:#2563eb; }

.cl-summary-full { color:#0c4a6e; font-size:13.5px; line-height:1.6; margin-top:8px; white-space:normal; }

/* Toggle logic */
.cl-summary:not([open]) .cl-summary-full { display:none; }
.cl-summary[open] .cl-summary-short { display:none; }
.cl-summary[open] .cl-more { display:none; }
.cl-summary:not([open]) .cl-less { display:none; }

/* Why details */
.cl-why summary { list-style:none; }
.cl-why summary::-webkit-details-marker { display:none; }
"""

with gr.Blocks(title="CollectiveLens", css=CSS) as demo:

    gr.HTML(
        """
        <div class="cl-hero">
            <h1>CollectiveLens</h1>
            <p>Your network’s knowledge, unified and searchable</p>
        </div>
        """
    )

    # Optional: paste API key for this session if env var isn't visible
    with gr.Accordion("⚙️ Session settings (optional)", open=False):
        api_key_box = gr.Textbox(
            label="Anthropic API key (used only in this session)",
            placeholder="Paste ANTHROPIC_API_KEY here if your environment variable isn't visible",
            type="password",
        )
        api_key_state = gr.State(value="")  # keep it in-memory for callbacks

        def set_key(k):
            return k or ""

        gr.Button("Use this key for this session").click(set_key, inputs=[api_key_box], outputs=[api_key_state])

    with gr.Tabs():
        # =========================
        # Search
        # =========================
        with gr.Tab("🔎 Search"):
            gr.Markdown("### Search the policy library")

            keyword_input = gr.Textbox(
                label="Search",
                placeholder="Search any language… (e.g., Erasmus, climate justice, student mental health)",
                lines=1
            )

            with gr.Row():
                topic_filter = gr.Dropdown(
                    choices=["All Topics"] + topic_list,
                    value="All Topics",
                    label="📚 Policy Area"
                )
                org_filter = gr.Dropdown(
                    choices=["All Organizations"] + org_list,
                    value=["All Organizations"],
                    label="🏢 Organization",
                    multiselect=True
                )

            period_choices = ["All years", "Last 3 years", "Last 5 years", f"{year_max-5}–{year_max}", "Custom range"]
            period = gr.Dropdown(choices=period_choices, value="All years", label="📅 Period")

            with gr.Row(visible=False) as custom_year_row:
                year_from_filter = gr.Dropdown(
                    choices=["All"] + [str(y) for y in range(year_min, year_max + 1)],
                    value="All",
                    label="From"
                )
                year_to_filter = gr.Dropdown(
                    choices=["All"] + [str(y) for y in range(year_min, year_max + 1)],
                    value="All",
                    label="To"
                )

            sort_filter = gr.Radio(
                choices=["Highest Relevance", "Most Recent"],
                value="Highest Relevance",
                label="Sort"
            )

            search_btn = gr.Button("🔍 Search", variant="primary")
            search_results = gr.HTML()

            def on_period_change(p):
                return gr.update(visible=(p == "Custom range"))

            period.change(on_period_change, inputs=[period], outputs=[custom_year_row])

            def run_search(keyword, topic, org, period_choice, year_from, year_to, sort_by, ui_key):
                yr_range, _ = period_to_year_range(period_choice, year_from, year_to)
                results = search_documents(keyword, topic, org, yr_range, sort_by)
                effective_key = get_effective_api_key(ui_key)
                return format_results(results, keyword, effective_key)

            search_btn.click(
                run_search,
                inputs=[keyword_input, topic_filter, org_filter, period, year_from_filter, year_to_filter, sort_filter, api_key_state],
                outputs=search_results
            )

        # =========================
        # Analytics
        # =========================
        with gr.Tab("📊 Analytics"):
            gr.Markdown("### Analytics dashboard")
            gr.Markdown("Filter by organization to explore patterns over time.")

            with gr.Row():
                org_selector = gr.Dropdown(
                    choices=["All Organizations"] + org_list,
                    value=["All Organizations"],
                    label="🏢 Organizations",
                    multiselect=True
                )
                update_btn = gr.Button("🔄 Update Dashboard", variant="primary")

            analytics_output = gr.HTML()
            update_btn.click(generate_stats, inputs=[org_selector], outputs=analytics_output)

    gr.HTML(
        f"""
        <div class="cl-footer">
            <div><strong>{len(df)}</strong> documents • <strong>{len(topic_list)}</strong> policy areas • <strong>{len(org_list)}</strong> organizations</div>
            <div style="margin-top:4px;opacity:0.85;">
                Powered by Claude (summaries & explanations) + SentenceTransformers (semantic search)
            </div>
        </div>
        """
    )

demo.launch(share=True, show_error=True)


ModuleNotFoundError: No module named 'gradio'